In [ ]:
#Check GPU
!nvidia-smi

Wed May 13 23:54:21 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   32C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
#Check CUDA
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [ ]:
#Check Build Tools
!cmake --version
!g++ --version
!python --version

cmake version 3.31.10

CMake suite maintained and supported by Kitware (kitware.com/cmake).
g++ (Ubuntu 11.4.0-1ubuntu1~22.04.3) 11.4.0
Copyright (C) 2021 Free Software Foundation, Inc.
This is free software; see the source for copying conditions.  There is NO
warranty; not even for MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE.

Python 3.12.13


In [ ]:
#Clone Repo to Colab
!git clone https://github.com/MighettoWasTaken/masters_project.git
%cd masters_project

Cloning into 'masters_project'...
remote: Enumerating objects: 1073, done.
remote: Counting objects: 100% (60/60), done.
remote: Compressing objects: 100% (52/52), done.
remote: Total 1073 (delta 8), reused 37 (delta 5), pack-reused 1013 (from 1)
Receiving objects: 100% (1073/1073), 19.24 MiB | 25.65 MiB/s, done.
Resolving deltas: 100% (475/475), done.
/content/masters_project


In [ ]:
#Install Dependencies
!pip install -e ".[dev]"

Obtaining file:///content/masters_project
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.9/91.9 kB 5.7 MB/s eta 0:00:00
  Using cached pathspec-1.1.1-py3-none-any.whl.metadata (14 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 47.7 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 78.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 254.6/254.6 kB 28.5 MB/s eta 0:00:00
Using cached pathspec-1.1.1-py3-none-any.whl (57 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 269.8/269.8 kB 30.1 MB/s eta 0:00:00
  Building editable for hodgkin-huxley (pyproject.toml) ... done
  Created wheel for hodgkin-huxley: filename=hodgkin_huxley-0.1.0-cp312-cp312-linux_x86_64.whl size=2523939 sha256=2a1e9d8fbc95d31dbbc112b39f6f91cd21cfef115ee69ab4140f

In [ ]:
#Verify Project
!python -c "from hodgkin_huxley import HHNeuron; n = HHNeuron(); print(n)"
!pytest tests/python/ -x -q


<HHNeuron V=-65.00 mV>
============================= test session starts ==============================
platform linux -- Python 3.12.13, pytest-8.4.2, pluggy-1.6.0
rootdir: /content/masters_project
configfile: pyproject.toml
plugins: cov-7.1.0, langsmith-0.7.34, typeguard-4.5.1, anyio-4.13.0
collected 1124 items                                                           

tests/python/test_api_cleanup.py ............................            [  2%]
tests/python/test_codegen.py ........................................... [  6%]
..............                                                           [  7%]
tests/python/test_composable_neuron.py ................................. [ 10%]
..............................................                           [ 14%]
tests/python/test_custom_equations.py .................................. [ 17%]
..............................                                           [ 20%]
tests/python/test_dbs_stimulator.py .............................

In [ ]:
%%writefile /tmp/test_cuda.cu
#include <cstdio>
__global__ void hello() {
    printf("Hello from GPU thread %d\n", threadIdx.x);
}
int main() {
    hello<<<1, 4>>>();
    cudaDeviceSynchronize();
    return 0;
}

Writing /tmp/test_cuda.cu


In [ ]:
# Compile and run
!nvcc /tmp/test_cuda.cu -o /tmp/test_cuda && /tmp/test_cuda


nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).
Hello from GPU thread 0
Hello from GPU thread 1
Hello from GPU thread 2
Hello from GPU thread 3


In [ ]:
# Run tests
!pytest tests/python/test_codegen.py::TestCUDAPrinter -v
